[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C35_Speech_Audio_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零实现**语音系统的核心机制，再与朴素参考或暴力枚举 **对拍**。

这个 notebook 做四件事：① 确认环境；② 体会「声音 = 一维时间序列」与采样率的含义；③ 体会本课贯穿的「丢相位」直觉；④ 立下全课纪律——**对拍（differential testing）**。

## 1 · 环境自检

只需要 `numpy`。`scipy`（个别对拍）与 `matplotlib`（画图）可选，缺失不影响课程。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
for opt in ['scipy', 'matplotlib']:
    try:
        m = __import__(opt); print(opt, getattr(m, '__version__', '?'), '(可选)')
    except Exception:
        print(opt, '未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 声音 = 一维时间序列

在计算机里，一段单声道音频就是一个一维数组 `x`，加上一个采样率 `sr`（每秒采样点数）。
**第 i 个采样的时刻 = i / sr 秒**。我们造一个 440 Hz 正弦（标准音 A）来体会这件事。

In [ ]:
sr = 16000                     # 采样率 16 kHz（语音/ASR 常用）
dur = 0.5                      # 0.5 秒
n = int(sr * dur)              # 采样点数
t = np.arange(n) / sr          # 每个采样的时刻（秒）
f0 = 440.0                     # 440 Hz = 标准音 A
x = np.sin(2 * np.pi * f0 * t) # 一维波形
print(f'采样率={sr} Hz, 时长={dur}s -> {n} 个采样点')
print(f'数组形状 {x.shape}, 取值范围 [{x.min():.2f}, {x.max():.2f}]')
# 一个周期含多少采样？= sr / f0
samples_per_cycle = sr / f0
print(f'440 Hz 每个周期约 {samples_per_cycle:.1f} 个采样')
assert x.shape == (n,)
assert abs(samples_per_cycle - 36.36) < 0.1
print('✅ 声音就是 (一维数组 + 采样率)；时刻 = 下标 / sr')

## 3 · 奈奎斯特：采样率决定能表示的最高频率

**采样定理**：采样率 `sr` 只能无失真表示低于 `sr/2`（奈奎斯特频率）的频率。

16 kHz 的奈奎斯特是 8 kHz——这就是为什么电话/ASR 用 16 kHz（人声主要能量在 8 kHz 以下），而音乐要 44.1 kHz（保留到 ~20 kHz 的高频泛音）。

In [ ]:
def nyquist(sr):
    return sr / 2

for sr_ in [8000, 16000, 44100, 48000]:
    print(f'采样率 {sr_:6d} Hz -> 奈奎斯特 {nyquist(sr_):8.1f} Hz（可表示的最高频率）')
assert nyquist(16000) == 8000
assert nyquist(44100) == 22050
print('\n✅ 采样率越高，能表示的频率越高，数据量也越大 —— 这是一个权衡')
print('语音 16 kHz 够用（人声 <8 kHz）；音乐要 44.1 kHz 留住高频泛音')

## 4 · 本课的核心直觉：人耳几乎不在乎相位

一个频率成分由 **幅度**（多强）和 **相位**（在周期里的位置）共同描述。

下面造两个**幅度谱完全相同、相位不同**的信号，验证它们波形不同——但人耳听起来几乎一样。这正是为什么语音特征普遍**丢掉相位、只留幅度**（谱图/梅尔/MFCC），也是模块 04 要把相位**造回来**的原因。

In [ ]:
N = 2048
t = np.arange(N) / sr
# 取频率正好落在 DFT 频点上(f=k*sr/N)，排除泄漏干扰，让幅度谱严格只由幅度决定
k1, k2 = 40, 90                       # 对应 312.5 Hz 与 703.1 Hz
f1, f2 = k1*sr/N, k2*sr/N
# 同样的两个频率、同样的幅度，但第二个加了相位偏移
x1 = np.sin(2*np.pi*f1*t) + 0.6*np.sin(2*np.pi*f2*t)
x2 = np.sin(2*np.pi*f1*t + 1.3) + 0.6*np.sin(2*np.pi*f2*t + 2.1)
# 它们的幅度谱（|FFT|）应完全相同（相位不进入幅度）
mag1 = np.abs(np.fft.rfft(x1))
mag2 = np.abs(np.fft.rfft(x2))
spec_diff = np.linalg.norm(mag1 - mag2) / np.linalg.norm(mag1)
wave_diff = np.linalg.norm(x1 - x2) / np.linalg.norm(x1)
print(f'幅度谱相对差异 = {spec_diff:.6f}  (≈0：幅度谱相同)')
print(f'波形  相对差异 = {wave_diff:.4f}  (很大：波形其实很不同)')
assert spec_diff < 1e-9, '频率对齐频点时，幅度谱应与相位无关'
assert wave_diff > 0.3, '波形应明显不同'
print('\n✅ 幅度谱相同、波形不同 —— 信息差就在相位里')
print('结论：丢相位很安全（听感几乎不变），但生成时要把它造回来（模块 04 Griffin-Lim）')

## 5 · 立纪律：对拍（differential testing）

本课每个「实现」都要和一个**绝对可信的参考**比对。先把工作流跑通：
从零写一个 DFT，对拍 numpy 的 `np.fft.fft`——这是模块 01 的第一个练习的预演。

In [ ]:
def dft_naive(x):
    '''从零实现离散傅里叶变换 X[k] = sum_n x[n] exp(-2j*pi*k*n/N)。'''
    x = np.asarray(x, dtype=complex)
    N = x.shape[0]
    k = np.arange(N)
    M = np.exp(-2j * np.pi * np.outer(k, k) / N)   # (N,N) DFT 矩阵
    return M @ x

rng = np.random.default_rng(0)
x = rng.standard_normal(32)
X_mine = dft_naive(x)
X_ref = np.fft.fft(x)
max_err = np.max(np.abs(X_mine - X_ref))
print(f'从零 DFT vs np.fft 最大误差 = {max_err:.2e}')
assert np.allclose(X_mine, X_ref, atol=1e-9), '从零 DFT 应等于 np.fft'
print('✅ 对拍通过：我的 DFT == np.fft（FFT 只是更快的同一个变换）')

## 6 · 一个会贯穿全课的对拍工具

把「对拍」封装成一个小函数，后面每个模块都用它判定「我的实现 == 参考」。它就是本课所有 `assert` 背后的统一裁判。

In [ ]:
def check_close(name, got, ref, atol=1e-8):
    '''对拍：被测实现结果 vs 可信参考。打印并 assert。'''
    got = np.asarray(got); ref = np.asarray(ref)
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

# 演示：分帧能量 == 直接逐元素平方和
def frame_energy(x, frame_len, hop):
    energies = []
    for start in range(0, len(x) - frame_len + 1, hop):
        frame = x[start:start+frame_len]
        energies.append(np.sum(frame**2))
    return np.array(energies)

x = rng.standard_normal(1000)
e = frame_energy(x, frame_len=100, hop=100)   # 不重叠时各帧能量应加起来=总能量
check_close('sum(frame energy) vs total', e.sum(), np.sum(x[:1000]**2))
print('\n这就是全课的工作流：写实现 -> 对拍可信参考 -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个特征/对齐/量化/重建实现，都会用 `np.allclose`（或单调性/暴力枚举）对拍可信参考；结构正确则数值一致，数值一致则逻辑可迁移到 torchaudio/Whisper/EnCodec。

**接下来五个模块**：01 信号与特征 → 02 ASR/CTC → 03 神经编解码 → 04 TTS/声码器 → 05 语音 LLM。每一步都建立在前一步之上（梅尔谱喂给 Whisper/声码器，token 化喂给语音 LM）。

下一站：**模块 01 · 信号与特征**。